# 10 — Persona theo cây quyết định

Giai đoạn này triển khai **cây 6 lá của notebook 09**. Chưa kết hợp CLV/affinity; lifecycle được giữ nguyên làm tag nguồn để dùng về sau, không ảnh hưởng phân nhóm cây.

**Nguồn:** output 08 bản event windows + output 05. Thêm vào Kaggle và chọn PERSONA_ROOT/LIFECYCLE_ROOT nếu có nhiều phiên bản. Giữ cùng tháng fit, seed, sample và feature như 09. Cây được fit một lần rồi áp dụng cố định; không học lại trên các tháng đánh giá hoặc tháng 05/2016.

Sáu nhóm dự kiến từ bản 09 đã chạy:
P1: NoEvent, 0–2 sản phẩm; P2: NoEvent, ≥3 sản phẩm; P3: HasEvent, F=1; P4: F≥2, 0–1 sản phẩm; P5: F≥2, 2–4 sản phẩm; P6: F≥2, ≥5 sản phẩm.

Notebook **kiểm tra cây fit lại có tương đương sáu rule này trên toàn miền F 0–17, số sản phẩm 0–24**. Nếu khác, dừng và yêu cầu xem lại nguồn/cấu hình; không gắn tên sáu nhóm cho một cây khác. R và exposure không có trong cây được chấp nhận. Vì không có model pickle của lần chạy 09, đây là tái dựng có kiểm chứng, không khẳng định byte model giống hệt.

Các tháng đánh giá đã được xem trước đây: kết quả vẫn là development backtest. F là số **tháng** có acquisition, không phải số sản phẩm mở. Chưa quan sát event không có nghĩa chưa bao giờ acquisition trong đời.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import silhouette_score, average_precision_score
PERSONA_ROOT=None  # folder chứa personas_YYYY-MM.parquet của 08
LIFECYCLE_ROOT=None  # folder chứa lifecycle_YYYY-MM.parquet của 05
FIT_MONTH='2015-11'
EVAL_MONTHS=['2015-12','2016-01','2016-02','2016-03','2016-04']
USE_M=False  # chỉ True khi xác nhận M upstream không dùng thông tin tương lai
K_OPTIONS=[3,5]
FIT_SAMPLE=100000
TREE_SAMPLE=200000
SEED=42
MIN_N=1000;MIN_POS=20
OUT=Path('/kaggle/working/persona_tree_outputs' if Path('/kaggle/working').exists() else './persona_tree_outputs')
OUT.mkdir(parents=True,exist_ok=True)
ROOT=Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('.')
def resolve(configured,pattern):
    root=Path(configured) if configured else ROOT
    parents=sorted({p.parent for p in root.rglob(pattern)})
    if len(parents)!=1:raise ValueError(f'{pattern}: đặt đường dẫn duy nhất, tìm thấy {parents}')
    return parents[0]
def shift(m,j):return str(pd.Period(m,'M')+j)
RECOMMEND_MONTH='2016-05'
P=resolve(PERSONA_ROOT,'personas_????-??.parquet')
L=resolve(LIFECYCLE_ROOT,'lifecycle_????-??.parquet')
assert FIT_MONTH<min(EVAL_MONTHS) and shift(FIT_MONTH,1)<=min(EVAL_MONTHS)
def load(m):
    d=pd.read_parquet(P/f'personas_{m}.parquet')
    needed=['customer_id','is_observed','products_complete','valid_pairs','R','F_event_months','n_products','event_history_group','persona']
    assert set(needed)<=set(d),'Cần output 08 bản event windows'
    assert d.customer_id.notna().all() and d.customer_id.is_unique
    d=d.loc[d.is_observed.eq(True)&d.products_complete.eq(True)].copy()
    d['branch']=np.where(d.F_event_months.eq(0),'NoEvent','HasEvent')
    d['exposure_rate']=d.F_event_months/d.valid_pairs.replace(0,np.nan)
    # Common cohort: đủ history, F/R hợp lệ; báo coverage riêng ngoài cohort.
    valid=d.valid_pairs.ge(3)&d.F_event_months.ge(0)&d.F_event_months.mod(1).eq(0)
    valid &= d.branch.eq('NoEvent') | (d.R.notna()&d.R.ge(0))
    valid &= d.n_products.between(0,24)&np.isfinite(d.exposure_rate)
    if USE_M:valid &= d.M_group_points.notna()&np.isfinite(d.M_group_points)&d.M_group_points.ge(0)
    return d.loc[valid].reset_index(drop=True),dict(month=m,current_complete_n=len(d),cohort_n=int(valid.sum()),excluded_n=int((~valid).sum()))
def target(d,m):
    z=pd.read_parquet(L/f'lifecycle_{shift(m,1)}.parquet',columns=['customer_id','acquisition_pair_valid','has_acquisition_month'])
    assert z.customer_id.is_unique
    assert not (z.acquisition_pair_valid.eq(True)&z.has_acquisition_month.isna()).any()
    j=d[['customer_id']].merge(z,on='customer_id',how='left',validate='one_to_one')
    valid=j.acquisition_pair_valid.eq(True).fillna(False).to_numpy(bool)
    y=j.has_acquisition_month.eq(True).fillna(False).to_numpy(int)
    return valid,y
fit,fit_coverage=load(FIT_MONTH)
fit_valid,fit_y=target(fit,FIT_MONTH)
print('Fit cohort',len(fit),'label coverage',fit_valid.mean(),'USE_M',USE_M)


## 1. Fit và xác nhận đúng sáu nhóm
Dùng tối đa 200.000 mẫu ngẫu nhiên tự nhiên tại 11/2015, nhãn 12/2015; max_leaf_nodes=6 và min_samples_leaf như 09. Nhãn này đã có tại cutoff đánh giá đầu 12/2015.

In [ ]:
TREE_COLS=['has_event','R_tree','F_event_months','n_products','valid_pairs','exposure_rate']
if USE_M:TREE_COLS.append('M_group_points')
def tree_frame(d):
    q=d.copy();q['has_event']=q.branch.eq('HasEvent').astype(int)
    q['R_tree']=q.R.where(q.has_event.eq(1),0)
    assert np.isfinite(q[TREE_COLS].to_numpy(float)).all()
    return q[TREE_COLS]
train=fit.loc[fit_valid].copy();train['y']=fit_y[fit_valid]
train=train.sample(min(TREE_SAMPLE,len(train)),random_state=SEED)
assert train.y.nunique()==2,'Cần đủ hai lớp tại tháng fit'
tree=DecisionTreeClassifier(max_leaf_nodes=6,min_samples_leaf=max(100,int(np.ceil(.01*len(train)))),random_state=SEED)
tree.fit(tree_frame(train),train.y)
rules=export_text(tree,feature_names=TREE_COLS,show_weights=True)
print(rules);(OUT/'tree_rules.txt').write_text(rules,encoding='utf-8')

NAMES={
'P1':'Danh mục cơ bản, chưa quan sát event',
'P2':'Danh mục đa sản phẩm, chưa quan sát event',
'P3':'Một tháng acquisition được quan sát',
'P4':'Acquisition lặp lại, danh mục 0–1 sản phẩm',
'P5':'Acquisition lặp lại, danh mục 2–4 sản phẩm',
'P6':'Acquisition lặp lại, danh mục ≥5 sản phẩm'}
RULES={
'P1':'F_event_months = 0; n_products <= 2',
'P2':'F_event_months = 0; n_products >= 3',
'P3':'F_event_months = 1',
'P4':'F_event_months >= 2; n_products <= 1',
'P5':'F_event_months >= 2; 2 <= n_products <= 4',
'P6':'F_event_months >= 2; n_products >= 5'}
def expected_persona(d):
    f=d.F_event_months;n=d.n_products
    labels=np.select([f.eq(0)&n.le(2),f.eq(0)&n.ge(3),f.eq(1),f.ge(2)&n.le(1),f.ge(2)&n.between(2,4),f.ge(2)&n.ge(5)],
        ['P1','P2','P3','P4','P5','P6'],default='INVALID')
    assert not (labels=='INVALID').any()
    return pd.Series(labels,index=d.index)
used={TREE_COLS[i] for i in tree.tree_.feature if i>=0}
assert used <= {'has_event','F_event_months','n_products'}, f'Cây khác bản đã chọn, dùng {used}; kiểm tra nguồn/cấu hình'
grid=pd.DataFrame([(f,c) for f in range(18) for c in range(25)],columns=['F_event_months','n_products'])
grid['branch']=np.where(grid.F_event_months.eq(0),'NoEvent','HasEvent')
grid['R']=0.;grid['valid_pairs']=17;grid['exposure_rate']=grid.F_event_months/17
grid['leaf']=tree.apply(tree_frame(grid));grid['persona_id']=expected_persona(grid)
audit=grid.groupby('leaf').persona_id.nunique()
assert len(audit)==6 and audit.eq(1).all() and grid.groupby('persona_id').leaf.nunique().eq(1).all(), 'Cây không tương đương sáu rule đã chọn'
LEAF_MAP=grid.drop_duplicates('leaf').set_index('leaf').persona_id.to_dict()
def assign_tree(d):
    d=d.copy();d['tree_leaf']=tree.apply(tree_frame(d))
    d['persona_id']=d.tree_leaf.map(LEAF_MAP)
    assert d.persona_id.eq(expected_persona(d)).all()
    d['persona_name']=d.persona_id.map(NAMES)
    d['persona_rule']=d.persona_id.map(RULES)
    d['persona_version']='tree6-nov2015-v1'
    return d
rule_table=pd.DataFrame([dict(persona_id=k,persona_name=NAMES[k],rule=RULES[k]) for k in NAMES])
rule_table.to_csv(OUT/'persona_rules.csv',index=False);display(rule_table)


## 2. Đánh giá từng nhóm qua tháng
Xác suất nhóm ước lượng từ toàn cohort fit đủ nhãn, smoothing prior_n=100; giữ cố định khi chấm các tháng sau. Báo rate thực tế, Wilson CI, lift nhóm so với baseline cùng cohort; phân biệt lift nhóm và Lift10 của toàn mô hình.

Khách không có nhãn hợp lệ không được điền target=0. Coverage có hai lớp: đủ điều kiện persona trong toàn bộ khách known; đủ nhãn tương lai trong persona cohort. Khoảng tin cậy từng tháng là mô tả, không coi các tháng của cùng khách là độc lập.

In [ ]:
def wilson(k,n):
    k=np.asarray(k,float);n=np.asarray(n,float);ok=n>0
    lo=np.full_like(n,np.nan);hi=lo.copy();pp=k[ok]/n[ok];nn=n[ok];z=1.96
    c=(pp+z*z/(2*nn))/(1+z*z/nn);h=z*np.sqrt(pp*(1-pp)/nn+z*z/(4*nn*nn))/(1+z*z/nn)
    lo[ok]=np.maximum(0,c-h);hi[ok]=np.minimum(1,c+h);return lo,hi
fit_assigned=assign_tree(fit)
rates=pd.DataFrame({'group':fit_assigned.persona_id,'valid':fit_valid,'y':fit_y})
rates=rates.loc[rates.valid].groupby('group').y.agg(['size','sum'])
prior=fit_y[fit_valid].mean()
score_map=(rates['sum']+100*prior)/(rates['size']+100)
rows=[];coverage=[];model_metrics=[];transition_parts=[];previous=None
for m in EVAL_MONTHS:
    d,cover=load(m);d=assign_tree(d);ok,y=target(d,m)
    assert ok.any()
    cover.update(label_n=int(ok.sum()),label_coverage=ok.mean());coverage.append(cover)
    d['label_valid']=ok;d['positive']=ok&(y==1);d['as_of_month']=m
    baseline=y[ok].mean()
    for pid,g in d.groupby('persona_id'):
        good=g.loc[g.label_valid];nn=len(good);kk=int(good.positive.sum())
        rows.append(dict(month=m,persona_id=pid,persona_name=NAMES[pid],n=len(g),share=len(g)/len(d),label_n=nn,positive_n=kk,
            rate=kk/nn if nn else np.nan,baseline=baseline,
            R_median=g.R.median() if g.R.notna().any() else np.nan,F_median=g.F_event_months.median(),
            products_median=g.n_products.median(),history_median=g.valid_pairs.median()))
    sc=d.persona_id.map(score_map).fillna(prior).to_numpy(float)[ok];yy=y[ok]
    k=max(1,int(.1*len(yy)));threshold=np.partition(sc,len(sc)-k)[len(sc)-k]
    high=sc>threshold;tie=sc==threshold
    precision=(yy[high].sum()+(k-high.sum())*yy[tie].mean())/k
    model_metrics.append(dict(month=m,AP=average_precision_score(yy,sc),Lift10=precision/baseline,Brier=np.mean((sc-yy)**2)))
    curr=d[['customer_id','persona_id']]
    if previous is not None:
        pm,prev=previous
        j=prev.merge(curr,on='customer_id',how='inner',validate='one_to_one',suffixes=('_from','_to'))
        t=j.groupby(['persona_id_from','persona_id_to']).size().rename('n').reset_index();t['source_month']=pm;t['target_month']=m
        transition_parts.append(t)
    previous=(m,curr.copy())
    d.drop(columns=['label_valid','positive']).to_parquet(OUT/f'personas_tree_{m}.parquet',index=False)
report=pd.DataFrame(rows);report['label_coverage']=report.label_n/report.n
report['lift']=report.rate/report.baseline
report['ci_low'],report['ci_high']=wilson(report.positive_n,report.label_n)
report['weak_evidence']=(report.label_n<MIN_N)|(report.positive_n<MIN_POS)
report.to_csv(OUT/'persona_validation_monthly.csv',index=False)
pd.DataFrame(coverage).to_csv(OUT/'cohort_label_coverage.csv',index=False)
pd.DataFrame(model_metrics).to_csv(OUT/'tree_model_metrics.csv',index=False)
display(report);display(pd.DataFrame(model_metrics))
stability=report.groupby(['persona_id','persona_name']).agg(months=('month','nunique'),min_rate=('rate','min'),max_rate=('rate','max'),min_lift=('lift','min'),max_lift=('lift','max'),min_label_n=('label_n','min'))
stability.to_csv(OUT/'persona_rate_stability.csv');display(stability)
plot=report.pivot(index='month',columns='persona_id',values='rate')
plot.plot(figsize=(11,5),marker='o',title='Acquisition tháng sau theo persona cây')
plt.ylabel('Tỷ lệ trên nhãn hợp lệ');plt.tight_layout();plt.savefig(OUT/'persona_rates.png',bbox_inches='tight');plt.show()
if transition_parts:
    transitions=pd.concat(transition_parts,ignore_index=True);transitions.to_csv(OUT/'persona_transition_counts.csv',index=False)
    counts=transitions.pivot_table(index='persona_id_from',columns='persona_id_to',values='n',aggfunc='sum',fill_value=0).reindex(index=NAMES,columns=NAMES,fill_value=0)
    transition_rate=counts.div(counts.sum(axis=1).replace(0,np.nan),axis=0)
    transition_rate.to_csv(OUT/'persona_transition_rates.csv');display(transition_rate)
    fig,ax=plt.subplots(figsize=(7,6));im=ax.imshow(transition_rate.fillna(0),vmin=0,vmax=1,cmap='Blues')
    ax.set_xticks(range(6),NAMES);ax.set_yticks(range(6),NAMES)
    for i in range(6):
        for j in range(6):
            v=transition_rate.iloc[i,j]
            ax.text(j,i,'NA' if pd.isna(v) else f'{v:.1%}',ha='center',va='center',color='white' if pd.notna(v) and v>.5 else 'black')
    ax.set(title='Chuyển nhóm — pooled customer-month, common cohort',xlabel='Persona tháng sau',ylabel='Persona tháng trước')
    fig.colorbar(im,ax=ax);plt.tight_layout();plt.savefig(OUT/'persona_transitions.png',bbox_inches='tight');plt.show()


## 3. Gán tại 05/2016, xuất thẻ và model
Không có nhãn 06/2016 nên không báo rate thực tế cho source 05/2016. Điểm nhóm là ước lượng khóa từ Nov→Dec, không phải dự báo đã calibration cho May. F=0 không gán R giả trong file output.

Mỗi known customer có một dòng coverage, nhưng chỉ khách đủ điều kiện mới có persona. Không gán nhóm giả cho khách vắng snapshot/incomplete/history ngắn. Giữ lifecycle và event_history_group nếu có để nối bước sau.

Hành động bên dưới là đề xuất thử nghiệm, chưa có CLV hay affinity để chỉ định sản phẩm cụ thể.

In [ ]:
import joblib
current,cover=load(RECOMMEND_MONTH);current=assign_tree(current)
current['as_of_month']=RECOMMEND_MONTH
current['persona_fit_score']=current.persona_id.map(score_map)
current.to_parquet(OUT/f'personas_tree_{RECOMMEND_MONTH}.parquet',index=False)
known=pd.read_parquet(P/f'personas_{RECOMMEND_MONTH}.parquet')
# Merge cờ nguồn cùng customer_id để không phụ thuộc index/thứ tự.
status=known[['customer_id','is_observed','products_complete']+[c for c in ['lifecycle'] if c in known]].copy()
status=status.merge(current[['customer_id','persona_id','persona_name']],on='customer_id',how='left',validate='one_to_one')
def coverage_status(frame):
    assigned=frame.persona_id.notna().to_numpy(dtype=bool)
    observed=frame.is_observed.astype('boolean')
    complete=frame.products_complete.astype('boolean')
    absent=observed.eq(False).to_numpy(dtype=bool,na_value=False)
    incomplete=(observed.eq(True)&~complete.fillna(False)).to_numpy(dtype=bool,na_value=False)
    unknown_observation=observed.isna().to_numpy(dtype=bool)
    return np.select([assigned,absent,incomplete,unknown_observation],
        ['Assigned','ExcludedNoCurrentSnapshot','ExcludedIncompleteSnapshot','ExcludedUnknownObservation'],
        default='ExcludedHistoryOrInvalidFeatures')
status['persona_status']=coverage_status(status)
assert len(status)==len(known) and status.customer_id.is_unique
status.to_parquet(OUT/f'persona_coverage_{RECOMMEND_MONTH}.parquet',index=False)
display(status.persona_status.value_counts())
ACTIONS={
'P1':'Kiểm tra nhu cầu cơ bản; thử tiếp cận chi phí thấp và đo hiệu quả.',
'P2':'Phân tích sản phẩm đang giữ ở bước affinity; không gọi là giá trị tiền cao.',
'P3':'Theo dõi acquisition tiếp theo; phân biệt R ở bước mô tả chi tiết.',
'P4':'Kiểm tra đóng/mở lại sản phẩm trước khi gọi là đang tăng trưởng danh mục.',
'P5':'Ứng viên thử bán chéo; lựa chọn sản phẩm cụ thể ở bước affinity.',
'P6':'Kiểm tra danh mục còn thiếu; không mặc định luôn có xác suất acquisition cao nhất.'}
profiles=[];cards=[]
for pid,g in current.groupby('persona_id'):
    historical=report.loc[report.persona_id.eq(pid)]
    row=dict(persona_id=pid,persona_name=NAMES[pid],rule=RULES[pid],n=len(g),share_eligible=len(g)/len(current),share_known=len(g)/len(known),
        F_median=g.F_event_months.median(),products_median=g.n_products.median(),
        historical_rate_min=historical.rate.min(),historical_rate_max=historical.rate.max(),
        action=ACTIONS[pid])
    profiles.append(row)
    cards.append(f"## {pid} — {NAMES[pid]}\n- Rule: {RULES[pid]}.\n- Quy mô: {len(g):,}; {len(g)/len(current):.1%} cohort đủ điều kiện, {len(g)/len(known):.1%} known customers.\n- Acquisition tháng sau trong development: {row['historical_rate_min']:.2%}–{row['historical_rate_max']:.2%}; không phải rate tương lai của tháng 05.\n- Đề xuất: {ACTIONS[pid]}\n")
pd.DataFrame(profiles).to_csv(OUT/'persona_profiles_latest.csv',index=False)
(OUT/'persona_cards.md').write_text('# Persona cây — '+RECOMMEND_MONTH+'\n\n'+'\n'.join(cards),encoding='utf-8')
joblib.dump({'tree':tree,'tree_columns':TREE_COLS,'leaf_map':LEAF_MAP,'names':NAMES,'rules':RULES,'score_map':score_map.to_dict(),'fit_month':FIT_MONTH,'seed':SEED},OUT/'persona_tree_model.joblib')
(OUT/'config.json').write_text(json.dumps(dict(version='tree6-nov2015-v1',fit_month=FIT_MONTH,fit_label_month=shift(FIT_MONTH,1),eval_months=EVAL_MONTHS,recommend_month=RECOMMEND_MONTH,seed=SEED,tree_sample=TREE_SAMPLE,USE_M=False,persona_root=str(P),lifecycle_root=str(L),scope='development; acquisition behavior only; CLV/affinity later'),ensure_ascii=False,indent=2),encoding='utf-8')
display(pd.DataFrame(profiles));print('Outputs:',OUT)

# Coverage bao gồm cả khách thiếu products_complete; không tự gán persona cho họ.


## Kết luận cần viết sau Run All
1. Sáu nhóm có quy mô, acquisition và độ tin cậy khác nhau thế nào?
2. P4 có phải reopen/đóng sản phẩm thường xuyên không? Hiện chưa đủ dữ liệu kết luận.
3. Lift/rate của mỗi nhóm giữ được qua các tháng không? Phân biệt ổn định nhãn và ổn định response.
4. Coverage loại khách nào? Không so với kết quả toàn population mà quên cohort khác.
5. Bước sau mới nối lifecycle, CLV và affinity trên customer_id + as_of_month; không refit cây chỉ để khớp insight mới.


## 4. Bổ sung bảng persona theo năm — dữ liệu từng phần
Giữ cây cố định. Hai bảng: năm tháng chốt, và năm acquisition tháng kế tiếp. Cutoff 12/2015 có outcome 01/2016. Backtest hiện chỉ có outcome Jan–May2016; không đủ để so acquisition giữa 2015 và 2016, không quy đổi thành cả năm.

Rate gộp = Σ positive_n / Σ label_n. Lift gộp so số event thực tế với Σ(label_n nhóm × baseline cùng tháng); tránh mix tháng khác nhau. Rate min/max là dao động của bình quân từng tháng, chưa phải khoảng tin cậy hay forecast. Quy mô cộng qua tháng là customer-month, không phải khách duy nhất. Không refit cây theo năm.

In [ ]:
def tree_yearly(frame,basis):
    d=frame.copy();periods=pd.PeriodIndex(d.month.astype(str),freq='M')
    d['cutoff_year']=periods.year;d['outcome_month']=(periods+1).astype(str);d['outcome_year']=(periods+1).year
    rows=[]
    for (year,pid),g in d.groupby([basis,'persona_id']):
        n=int(g.label_n.sum());pos=int(g.positive_n.sum())
        expected=(g.label_n*g.baseline).sum()
        rows.append(dict(year_basis=basis,year=int(year),persona_id=pid,persona_name=g.persona_name.iloc[0],
            cutoff_months=', '.join(sorted(g.month.unique())),outcome_months=', '.join(sorted(g.outcome_month.unique())),
            evaluated_months=g.month.nunique(),partial_year=True,
            customer_cutoff_rows=int(g.n.sum()),labelled_customer_cutoff_rows=n,positive_customer_cutoff_rows=pos,
            label_coverage=n/g.n.sum() if g.n.sum() else np.nan,
            acquisition_rate_pooled=pos/n if n else np.nan,
            acquisition_rate_month_min=g.rate.min(),acquisition_rate_month_max=g.rate.max(),
            lift_macro=g.lift.mean(),lift_month_min=g.lift.min(),lift_month_max=g.lift.max(),
            lift_mix_adjusted=pos/expected if expected else np.nan,
            strong_months=int((~g.weak_evidence).sum())))
    return pd.DataFrame(rows)
for basis in ['cutoff_year','outcome_year']:
    yearly=tree_yearly(report,basis)
    yearly.to_csv(OUT/f'persona_by_{basis}.csv',index=False)
    print(basis,'— chỉ các tháng đã có nhãn, không phải báo cáo đủ cả năm')
    display(yearly.round(4))


## 5. RFM có phân biệt thêm bên trong sáu persona không?
Giữ nguyên P1–P6 và cây. So PersonaOnly với Persona+R, +F4, +M và +RFM, cùng cohort và target có acquisition tháng sau. Chỉ bổ sung **tag**, chưa đổi persona chính.

Thêm output notebook 03 có bucket_specs.json + config.json. Mặc định `Hybrid_REDA_F4_M5` (F4: 1 / 2 / 3–4 / ≥5); sửa RFM_METHOD để dùng phương án đã chốt khác. Biên lấy nguyên từ 03, không fit lại theo evaluation. Fit biên phải không muộn hơn FIT_MONTH của cây. Tái tạo nhãn để dùng cả 05/2016, vì assignments 03 thường chỉ xuất đến 04/2016.

**M là propensity proxy, không phải tiền.** Phải kiểm tra M upstream chỉ dùng thông tin đã có tại cutoff; kiểm tra fit_month của 03 chưa tự bảo đảm M không dùng nhãn tương lai. Nếu chưa xác nhận được, các kết luận +M/+RFM chỉ là exploratory có điều kiện. +R/+F là các ablation tách riêng để không dựa hoàn toàn vào M.

NoEvent không gán R giả: dùng NoObservedEvent cho RF và phân tích M riêng bằng cùng thang M đã khóa (thang này học trên HasEvent nên không gọi là quantile riêng của NoEvent).

Score nhóm học từ Nov→Dec, khóa cho các tháng sau. Nhóm refined shrink về rate persona, không về rate tính từ tháng evaluation. Mọi phương án dùng cùng khách, kể cả thiếu M (tag UnknownM); báo coverage rõ. Các tháng đã xem là development, không dùng bảng để tuyên bố thắng ngoài mẫu mới.

In [ ]:
RFM_MODEL_ROOT=None
RFM_METHOD='Hybrid_REDA_F4_M5'
RFM_MIN_N=100
RFM_MIN_POS=20
RFM_PRIOR_N=100
RFM_OUT=OUT/'rfm_audit';RFM_OUT.mkdir(exist_ok=True)
RFM_ROOT=resolve(RFM_MODEL_ROOT,'bucket_specs.json')
rfm_specs=json.loads((RFM_ROOT/'bucket_specs.json').read_text())
rfm_cfg=json.loads((RFM_ROOT/'config.json').read_text())
assert RFM_METHOD in rfm_specs,f'Không có {RFM_METHOD}; lựa chọn: {list(rfm_specs)}'
assert pd.Period(str(rfm_cfg['fit_month'])[:7],'M')<=pd.Period(FIT_MONTH,'M')
assert rfm_cfg['features']==['R','F_event_months','M_group_points']
edges={c:np.asarray(rfm_specs[RFM_METHOD][c],float) for c in rfm_cfg['features']}
assert all(len(e)>=2 and (np.diff(e)>0).all() and np.isneginf(e[0]) and np.isposinf(e[-1]) for e in edges.values())
print('RFM method:',RFM_METHOD,'fit edges:',rfm_cfg['fit_month'])
print('Lưu ý: kiểm tra M upstream as-of trước khi dùng kết luận +M/+RFM.')

def rfm_tags(d):
    d=assign_tree(d)
    assert 'M_group_points' in d,'Output 08 thiếu M_group_points'
    f=pd.to_numeric(d.F_event_months,errors='coerce');m=pd.to_numeric(d.M_group_points,errors='coerce')
    valid_m=m.notna()&np.isfinite(m)&m.ge(0)
    for col,prefix in [('R','R'),('F_event_months','F'),('M_group_points','M')]:
        v=pd.to_numeric(d[col],errors='coerce')
        codes=pd.cut(v,edges[col],labels=False,right=True)
        d[prefix+'_tag']=codes.map(lambda z:prefix+str(int(z)+1) if pd.notna(z) else 'Unknown'+prefix)
    d.loc[~valid_m,'M_tag']='UnknownM'
    d.loc[f.eq(0),['R_tag','F_tag']]='NoObservedEvent'
    d['M_valid']=valid_m
    d['rfm_tag']=np.where(f.eq(0),'NoObservedEvent_'+d.M_tag,d.R_tag+'_'+d.F_tag+'_'+d.M_tag)
    return d

def group_keys(d,method):
    base=d.persona_id.astype(str)
    if method=='PersonaOnly':return base
    if method=='Persona_R':return base+' | '+d.R_tag
    if method=='Persona_F4':return base+' | '+d.F_tag
    if method=='Persona_M':return base+' | '+d.M_tag
    if method=='Persona_RFM':return base+' | '+d.rfm_tag
    raise ValueError(method)
RFM_METHODS=['PersonaOnly','Persona_R','Persona_F4','Persona_M','Persona_RFM']
rf=rfm_tags(fit);ok,yy=target(rf,FIT_MONTH)
parent=pd.DataFrame({'persona_id':rf.persona_id,'y':yy,'ok':ok})
parent=parent.loc[parent.ok].groupby('persona_id').y.agg(['size','sum'])
prior=float(yy[ok].mean());parent_score=(parent['sum']+RFM_PRIOR_N*prior)/(parent['size']+RFM_PRIOR_N)
locked_scores={};fit_support={}
for method in RFM_METHODS:
    keys=group_keys(rf,method)
    z=pd.DataFrame({'group':keys,'persona_id':rf.persona_id,'y':yy,'ok':ok})
    z=z.loc[z.ok].groupby('group').agg(n=('y','size'),positive_n=('y','sum'),persona_id=('persona_id','first'))
    if method=='PersonaOnly':scores=parent_score
    else:scores=(z.positive_n+RFM_PRIOR_N*z.persona_id.map(parent_score))/(z.n+RFM_PRIOR_N)
    locked_scores[method]=scores;fit_support[method]=z
    z.assign(score=scores).to_csv(RFM_OUT/f'fit_support_{method}.csv')

def rfm_metrics(y,score):
    y=np.asarray(y,int);score=np.asarray(score,float);base=y.mean();k=max(1,int(np.ceil(.1*len(y))))
    cut=np.partition(score,len(score)-k)[len(score)-k];high=score>cut;tie=score==cut
    precision=(y[high].sum()+(k-high.sum())*y[tie].mean())/k
    return dict(n=len(y),baseline=base,AP=average_precision_score(y,score) if y.sum() else np.nan,
        Lift10=precision/base if base else np.nan,Brier=np.mean((y-score)**2),score_distinct=len(np.unique(score)))
metric_rows=[];profile_rows=[];cover_rows=[]
for month in EVAL_MONTHS:
    d,a=load(month);d=rfm_tags(d);valid,y=target(d,month);assert valid.any()
    cover_rows.append(dict(month=month,n=len(d),label_n=int(valid.sum()),M_valid_n=int(d.M_valid.sum()),M_coverage=d.M_valid.mean()))
    d['label_valid']=valid;d['positive']=valid&(y==1)
    # Parent outcome chỉ làm baseline mô tả tại tháng này, không dùng làm score.
    parent_actual=d.loc[valid].groupby('persona_id').positive.mean()
    for method in RFM_METHODS:
        keys=group_keys(d,method);rawscore=keys.map(locked_scores[method]);unseen=rawscore.isna()
        scores=rawscore.fillna(d.persona_id.map(parent_score)).fillna(prior).to_numpy(float)
        metric_rows.append(dict(month=month,persona_id='ALL',method=method,fallback_share=float(unseen[valid].mean()),**rfm_metrics(y[valid],scores[valid])))
        for pid,ix in d.groupby('persona_id').indices.items():
            indices=np.asarray(ix);eligible=indices[valid[indices]]
            if len(eligible):metric_rows.append(dict(month=month,persona_id=pid,method=method,fallback_share=float(unseen.iloc[eligible].mean()),**rfm_metrics(y[eligible],scores[eligible])))
        for group,ix in keys.groupby(keys).groups.items():
            g=d.loc[ix];eligible=g.loc[g.label_valid];count=len(eligible);positive=int(eligible.positive.sum());pid=g.persona_id.iloc[0]
            rate=positive/count if count else np.nan
            profile_rows.append(dict(month=month,method=method,group=group,persona_id=pid,n=len(g),label_n=count,positive_n=positive,
                rate=rate,parent_rate=parent_actual.get(pid,np.nan),lift_vs_parent=rate/parent_actual[pid] if count and parent_actual.get(pid,0)>0 else np.nan,
                share_within_persona=len(g)/int(d.persona_id.eq(pid).sum()),
                weak_evidence=count<RFM_MIN_N or positive<RFM_MIN_POS))
metrics_rfm=pd.DataFrame(metric_rows);profiles_rfm=pd.DataFrame(profile_rows)
metrics_rfm.to_csv(RFM_OUT/'predictive_comparison_monthly.csv',index=False)
profiles_rfm['ci_low'],profiles_rfm['ci_high']=wilson(profiles_rfm.positive_n,profiles_rfm.label_n)
profiles_rfm.to_csv(RFM_OUT/'subgroup_acquisition_monthly.csv',index=False)
pd.DataFrame(cover_rows).to_csv(RFM_OUT/'rfm_coverage.csv',index=False)
base=metrics_rfm.loc[metrics_rfm.method.eq('PersonaOnly'),['month','persona_id','AP','Lift10','Brier']].rename(columns={c:'parent_'+c for c in ['AP','Lift10','Brier']})
comp=metrics_rfm.merge(base,on=['month','persona_id'],validate='many_to_one')
for col in ['AP','Lift10','Brier']:comp['delta_'+col]=comp[col]-comp['parent_'+col]
summary_rfm=comp.groupby(['persona_id','method']).agg(months=('month','nunique'),AP_mean=('AP','mean'),Lift10_mean=('Lift10','mean'),Lift10_min=('Lift10','min'),Brier_mean=('Brier','mean'),delta_AP_mean=('delta_AP','mean'),delta_Lift10_mean=('delta_Lift10','mean'),delta_Brier_mean=('delta_Brier','mean'),fallback_max=('fallback_share','max')).reset_index()
summary_rfm.to_csv(RFM_OUT/'incremental_value_summary.csv',index=False)
display(summary_rfm.loc[summary_rfm.persona_id.eq('ALL')].round(4))
print('Trong từng persona: baseline PersonaOnly có cùng score trong nhóm, Lift10 kỳ vọng=1 nếu có positives.')
display(summary_rfm.loc[~summary_rfm.persona_id.eq('ALL')].round(4))
# Support + ổn định response của subgroup qua tháng.
stable=profiles_rfm.groupby(['method','persona_id','group']).agg(months=('month','nunique'),strong_months=('weak_evidence',lambda s:int((~s).sum())),min_label_n=('label_n','min'),rate_min=('rate','min'),rate_max=('rate','max'),lift_parent_min=('lift_vs_parent','min'),lift_parent_max=('lift_vs_parent','max')).reset_index()
stable.to_csv(RFM_OUT/'subgroup_rate_stability.csv',index=False)
latest_rf=rfm_tags(load(RECOMMEND_MONTH)[0]);latest_rf['as_of_month']=RECOMMEND_MONTH
latest_rf.to_parquet(RFM_OUT/f'persona_rfm_tags_{RECOMMEND_MONTH}.parquet',index=False)
counts=pd.crosstab(latest_rf.persona_id,latest_rf.rfm_tag)
counts.to_csv(RFM_OUT/'persona_rfm_crosstab_latest.csv')
print('Ví dụ subgroup đủ support, tháng cuối:')
view=profiles_rfm.loc[profiles_rfm.month.eq(EVAL_MONTHS[-1])&profiles_rfm.method.eq('Persona_RFM')&~profiles_rfm.weak_evidence]
display(view.sort_values(['persona_id','rate'],ascending=[True,False]).groupby('persona_id').head(5))
(RFM_OUT/'config.json').write_text(json.dumps(dict(rfm_method=RFM_METHOD,rfm_root=str(RFM_ROOT),edge_fit_month=rfm_cfg['fit_month'],score_fit_month=FIT_MONTH,eval_months=EVAL_MONTHS,refit_persona=False,rate_prior_n=RFM_PRIOR_N,scope='development; verify upstream M as-of; no automatic subgroup promotion'),ensure_ascii=False,indent=2),encoding='utf-8')
print('RFM audit outputs:',RFM_OUT)


### Kết luận sau Run All
1. RFM chỉ thêm độ chi tiết hay giúp ranking thực tế? Đọc delta AP/Lift10 và Brier so PersonaOnly trên cùng khách.
2. +R, +F4, +M: chiều nào bổ sung rõ nhất? Nếu chỉ +M cải thiện, phải kiểm tra temporal validity của M trước khi kết luận.
3. Trong P4/P5/P6, subgroup nào đủ mẫu và giữ lift_vs_parent khác 1 qua nhiều tháng? Không chọn từ một rate cực trị duy nhất.
4. P1/P2 không có R hợp lệ; phần phân biệt thêm chủ yếu là M. Không ép NoEvent vào R=0.
5. Nếu RFM không cải thiện bền, giữ persona cây và dùng RFM như tag mô tả. Nếu có subgroup rõ, mới cân nhắc tách thêm; cần giai đoạn mới để xác nhận vì đây là development.

Persona vẫn là P1–P6. File persona_rfm_tags_2016-05.parquet chỉ bổ sung tag và giữ key customer_id/as_of_month, chưa thay output chính dùng bởi notebook 11/12.

## 6. Bảng range acquisition và độ chồng lấp — không thêm biểu đồ
Biểu đồ acquisition ở trên là **tỷ lệ có acquisition tháng sau**, không phải lift hay CLV score. Bảng này trình bày lại dưới dạng range để xem nhóm có dao động rộng và có giao nhau không.

Range = min–max tỷ lệ bình quân persona qua các cutoff đã đánh giá. Đây không phải range hành vi từng khách, không phải khoảng tin cậy và không phải biên đảm bảo tương lai. Width tính bằng **điểm phần trăm (pp)**. Ví dụ 10%→15% rộng 5 pp.

Một nhóm có tháng mùa vụ tăng cao có thể làm range giao nhóm khác dù thứ tự ở từng tháng vẫn giữ nguyên. Vì vậy có thêm bảng so cặp **cùng tháng**: tỷ lệ A>B bao nhiêu tháng, gap min–max, số tháng CI Wilson giao nhau. Không suy ra kiểm định thống kê chỉ từ CI có/không giao. Chỉ có 5 tháng, chưa đủ gọi range là giới hạn dài hạn.

Không có ngưỡng phổ quát cho “quá rộng”. Đọc width cùng mean, std và mẫu theo tháng; không tự động tách persona chỉ vì một tháng biến động. RFM-audit có range subgroup tại subgroup_rate_stability.csv; phần này tập trung P1–P6 của hình.

In [ ]:
from itertools import combinations

def persona_range_tables(data):
    d=data.copy()
    assert not d.duplicated(['month','persona_id']).any()
    a=d.groupby('persona_id').agg(persona_name=('persona_name','first'),months=('month','nunique'),
        acquisition_min=('rate','min'),acquisition_mean=('rate','mean'),acquisition_max=('rate','max'),
        acquisition_std=('rate','std'),min_label_n=('label_n','min'),max_label_n=('label_n','max'))
    a['width_pp']=(a.acquisition_max-a.acquisition_min)*100
    a['std_pp']=a.acquisition_std*100
    a['width_over_mean']=a.width_pp/(a.acquisition_mean*100).replace(0,np.nan)
    pair_rows=[]
    for pa,pb in combinations(sorted(a.index),2):
        amin,amax=a.loc[pa,['acquisition_min','acquisition_max']]
        bmin,bmax=a.loc[pb,['acquisition_min','acquisition_max']]
        low=max(amin,bmin);high=min(amax,bmax);overlap=max(0.,high-low)
        shorter=min(amax-amin,bmax-bmin)
        union=max(amax,bmax)-min(amin,bmin)
        joined=d.loc[d.persona_id.eq(pa),['month','rate','ci_low','ci_high']].merge(
            d.loc[d.persona_id.eq(pb),['month','rate','ci_low','ci_high']],on='month',validate='one_to_one',suffixes=('_A','_B'))
        gap=joined.rate_A-joined.rate_B
        ci_valid=joined[['ci_low_A','ci_high_A','ci_low_B','ci_high_B']].notna().all(axis=1)
        ci_touch=(joined.ci_low_A<=joined.ci_high_B)&(joined.ci_low_B<=joined.ci_high_A)&ci_valid
        pair_rows.append(dict(persona_A=pa,persona_B=pb,range_A_min=amin,range_A_max=amax,range_B_min=bmin,range_B_max=bmax,
            ranges_intersect=bool(high>=low),overlap_min=low if high>=low else np.nan,overlap_max=high if high>=low else np.nan,
            overlap_width_pp=overlap*100,
            overlap_fraction_shorter=overlap/shorter if shorter>0 else np.nan,
            overlap_fraction_union=overlap/union if union>0 else np.nan,
            common_months=len(joined),A_above_B_months=int(gap.gt(0).sum()),B_above_A_months=int(gap.lt(0).sum()),
            same_month_gap_min_pp=gap.min()*100,same_month_gap_max_pp=gap.max()*100,
            valid_CI_months=int(ci_valid.sum()),CI_intersection_months=int(ci_touch.sum())))
    return a.reset_index(),pd.DataFrame(pair_rows)

range_summary,range_pairs=persona_range_tables(report)
RANGE_OUT=OUT/'range_tables';RANGE_OUT.mkdir(exist_ok=True)
range_summary.to_csv(RANGE_OUT/'persona_acquisition_ranges.csv',index=False)
range_pairs.to_csv(RANGE_OUT/'persona_range_overlap_pairs.csv',index=False)
# Display percentages explicitly, keep numeric CSV for reuse.
show=range_summary[['persona_id','months','acquisition_min','acquisition_mean','acquisition_max','width_pp','std_pp','min_label_n']].copy()
for c in ['acquisition_min','acquisition_mean','acquisition_max']:show[c]=show[c].map(lambda x:f'{x:.2%}')
show[['width_pp','std_pp']]=show[['width_pp','std_pp']].round(2)
print('Tỷ lệ acquisition tháng sau: range của bình quân từng tháng, width tính bằng pp')
display(show)
print('Các cặp có range giao nhau (không nhất thiết đảo thứ tự cùng tháng):')
display(range_pairs.loc[range_pairs.ranges_intersect,['persona_A','persona_B','overlap_min','overlap_max','overlap_width_pp',
    'overlap_fraction_shorter','common_months','A_above_B_months','B_above_A_months','same_month_gap_min_pp','same_month_gap_max_pp','CI_intersection_months']].round(4))
print('Toàn bộ 15 cặp, kể cả không giao nhau:')
display(range_pairs[['persona_A','persona_B','ranges_intersect','overlap_width_pp','common_months','A_above_B_months','B_above_A_months','CI_intersection_months']])
# Readable text ranges for quick reporting.
for _,row in range_summary.iterrows():
    print(f"{row.persona_id}: {row.acquisition_min:.2%}–{row.acquisition_max:.2%}; rộng {row.width_pp:.2f} pp, {int(row.months)} tháng.")


## 7. Bảng ngang từng tháng — range có dấu " - "
Hai bảng, không thêm biểu đồ:
1. Tỷ lệ acquisition thực tế từng tháng, kèm min–max các tỷ lệ qua tháng.
2. Khoảng tin cậy Wilson 95% của tỷ lệ từng tháng, mỗi ô dạng **cận dưới% - cận trên%**. Min–max tổng hợp là range bình quân tháng, không phải CI chung.

Các cột là **tháng chốt**, đo acquisition ở tháng kế tiếp. Một khách có outcome nhị phân 0/1; không có một range cá nhân liên tục để lấy min–max trong tháng. CI thể hiện độ bất định của tỷ lệ nhóm, không phải phân bố xác suất giữa từng khách.

Thêm bảng chi tiết 15 cặp persona theo tháng: khoảng CI, chênh lệch pp và sample. CI giao nhau không phải phép kiểm định; không giao nhau cũng chưa chứng minh ý nghĩa kinh doanh. Không so CI của các tháng khác nhau rồi kết luận hai persona không phân biệt được.

In [ ]:
def monthly_range_tables(data):
    d=data.copy();assert not d.duplicated(['persona_id','month']).any()
    months_order=sorted(d.month.astype(str).unique())
    rates=d.pivot(index='persona_id',columns='month',values='rate').reindex(columns=months_order)
    rates_fmt=rates.map(lambda x:'NA' if pd.isna(x) else f'{x:.2%}')
    ranges=rates.apply(lambda row:f'{row.min():.2%} - {row.max():.2%}' if row.notna().any() else 'NA',axis=1)
    rates_fmt['Min - max qua tháng']=ranges
    d['CI95_range']=d.apply(lambda row:'NA' if pd.isna(row.ci_low) or pd.isna(row.ci_high) else f'{row.ci_low:.2%} - {row.ci_high:.2%}',axis=1)
    ci=d.pivot(index='persona_id',columns='month',values='CI95_range').reindex(columns=months_order).fillna('NA')
    ci['Min - max tỷ lệ qua tháng (không phải CI)']=ranges
    pair_rows=[]
    from itertools import combinations
    for month,g in d.groupby('month'):
        for pa,pb in combinations(sorted(g.persona_id.unique()),2):
            a=g.loc[g.persona_id.eq(pa)].iloc[0];b=g.loc[g.persona_id.eq(pb)].iloc[0]
            has_ci=pd.notna(a.ci_low) and pd.notna(a.ci_high) and pd.notna(b.ci_low) and pd.notna(b.ci_high)
            pair_rows.append(dict(cutoff_month=month,outcome_month=str(pd.Period(month,'M')+1),persona_A=pa,persona_B=pb,
                rate_A=a.rate,rate_B=b.rate,CI95_A=a.CI95_range,CI95_B=b.CI95_range,
                gap_A_minus_B_pp=(a.rate-b.rate)*100,label_n_A=a.label_n,label_n_B=b.label_n,
                CI_intersects=bool(max(a.ci_low,b.ci_low)<=min(a.ci_high,b.ci_high)) if has_ci else pd.NA,
                weak_A=bool(a.weak_evidence),weak_B=bool(b.weak_evidence)))
    return rates_fmt,ci,pd.DataFrame(pair_rows)

monthly_rates_display,monthly_CI_display,monthly_pairs=monthly_range_tables(report)
MONTH_OUT=OUT/'range_tables';MONTH_OUT.mkdir(exist_ok=True)
monthly_rates_display.to_csv(MONTH_OUT/'persona_acquisition_monthly_wide.csv')
monthly_CI_display.to_csv(MONTH_OUT/'persona_CI95_monthly_ranges.csv')
monthly_pairs.to_csv(MONTH_OUT/'persona_pairs_by_month.csv',index=False)
print('Tỷ lệ acquisition: cột là tháng chốt, outcome ở tháng tiếp theo')
display(monthly_rates_display)
print('Range từng tháng = CI Wilson 95%; dấu " - " giữa hai cận')
display(monthly_CI_display)
print('Chi tiết cặp persona tháng cuối:')
display(monthly_pairs.loc[monthly_pairs.cutoff_month.eq(sorted(report.month.unique())[-1])])


## 8. Decision tree chỉ dùng F và M — không dùng R
Thử nghiệm mới chỉ dùng F_event_months và M_group_points. NoEvent giữ riêng F=0, chia theo M; HasEvent chia theo F/M. F-only là ablation. Baseline FM_F4_M5 dùng biên F4/M5 frozen từ notebook 03 và bỏ R khi tạo nhãn.

Giữ cohort của thí nghiệm cũ để đối chiếu: load vẫn kiểm tra chất lượng lịch sử/R cho HasEvent, nhưng R không được đưa vào cây, score, nhãn FM hay bảng feature range. P1–P6 và các phần 1–7 là kết quả lịch sử, giữ nguyên. Phần mới xuất sang fm_decision_tree để không trộn output cũ.

Fit Nov→Dec, chọn số nhóm bằng validation Dec→Jan; đánh giá Jan–Apr, không refit cây theo tháng. M là propensity proxy; phải xác nhận weights upstream chỉ dùng lịch sử khả dụng. Các tháng đã được xem là development. Chạy phần trước để có load, target, edges, wilson, rfm_metrics. Bảng mới đã xóa output cũ, cần chạy lại.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
import joblib
DT_OUT=OUT/'fm_decision_tree';DT_OUT.mkdir(exist_ok=True)
DT_MAX_LEAVES_PER_BRANCH=3
DT_MIN_LEAF_SHARE=.01
DT_PRIOR_N=100
DT_MODELS={};DT_MAPS={};DT_SCORES={};DT_PRIORS={}

def dt_common(month):
    d,coverage=load(month)
    m=pd.to_numeric(d.M_group_points,errors='coerce')
    good=m.notna()&np.isfinite(m)&m.ge(0)
    coverage.update(before_M_n=len(d),valid_M_n=int(good.sum()),M_coverage=float(good.mean()))
    return d.loc[good].reset_index(drop=True),coverage

def dt_hybrid(d):
    tags={}
    for f,prefix in [('F_event_months','F'),('M_group_points','M')]:
        tags[prefix]=pd.cut(d[f],edges[f],labels=False,include_lowest=True).map(lambda x:prefix+str(int(x)+1) if pd.notna(x) else 'Unknown'+prefix)
    keys=tags['F']+'_'+tags['M']
    no=d.F_event_months.eq(0)
    keys.loc[no]='NoObservedEvent_'+tags['M'][no]
    return keys

def dt_assign(d,method):
    if method=='FM_F4_M5':return dt_hybrid(d)
    result=pd.Series(index=d.index,dtype='string')
    for branch,mask in [('NoEvent',d.F_event_months.eq(0)),('HasEvent',d.F_event_months.gt(0))]:
        if not mask.any():continue
        spec=DT_MODELS[(method,branch)]
        if spec['tree'] is None:result.loc[mask]=method+'_'+branch+'_G1'
        else:
            leaves=spec['tree'].apply(d.loc[mask,spec['features']])
            result.loc[mask]=pd.Series(leaves,index=d.index[mask]).map(DT_MAPS[(method,branch)])
    assert result.notna().all()
    return result

dt_fit,dt_fit_coverage=dt_common(FIT_MONTH)
dt_ok,dt_y=target(dt_fit,FIT_MONTH)
assert dt_ok.any()
rule_rows=[]
for method in ['F_tree','FM_tree']:
    for branch,mask in [('NoEvent',dt_fit.F_event_months.eq(0)),('HasEvent',dt_fit.F_event_months.gt(0))]:
        columns=(['M_group_points'] if method=='FM_tree' else []) if branch=='NoEvent' else (['F_event_months','M_group_points'] if method=='FM_tree' else ['F_event_months'])
        train=mask.to_numpy()&dt_ok
        assert train.sum()>0,f'Không có nhãn fit: {branch}'
        tree=None
        if columns:
            tree=DecisionTreeClassifier(max_leaf_nodes=DT_MAX_LEAVES_PER_BRANCH,
                min_samples_leaf=max(100,int(np.ceil(train.sum()*DT_MIN_LEAF_SHARE))),random_state=42)
            tree.fit(dt_fit.loc[train,columns],dt_y[train])
            leaves=sorted(set(tree.apply(dt_fit.loc[train,columns])))
            DT_MAPS[(method,branch)]={leaf:f'{method}_{branch}_G{i+1}' for i,leaf in enumerate(leaves)}
            (DT_OUT/f'{method}_{branch}_tree.txt').write_text(export_text(tree,feature_names=columns,decimals=6))
            def walk(node,conditions):
                left,right=tree.tree_.children_left[node],tree.tree_.children_right[node]
                if left==right:
                    rule_rows.append(dict(method=method,group=DT_MAPS[(method,branch)][node],branch=branch,
                        rule=' AND '.join(conditions),fit_n=int(tree.tree_.n_node_samples[node])))
                else:
                    feature=columns[tree.tree_.feature[node]];threshold=tree.tree_.threshold[node]
                    walk(left,conditions+[f'{feature} <= {threshold:.9g}'])
                    walk(right,conditions+[f'{feature} > {threshold:.9g}'])
            walk(0,['F_event_months = 0' if branch=='NoEvent' else 'F_event_months >= 1'])
        else:rule_rows.append(dict(method=method,group=method+'_'+branch+'_G1',branch=branch,rule='F_event_months = 0',fit_n=int(train.sum())))
        DT_MODELS[(method,branch)]=dict(tree=tree,features=columns)
DT_METHODS=['FM_F4_M5','F_tree','FM_tree']
for method in DT_METHODS:
    keys=dt_assign(dt_fit,method)
    z=pd.DataFrame(dict(group=keys,y=dt_y,valid=dt_ok))
    rates=z.loc[z.valid].groupby('group').y.agg(['size','sum'])
    prior=float(dt_y[dt_ok].mean());DT_PRIORS[method]=prior
    DT_SCORES[method]=(rates['sum']+DT_PRIOR_N*prior)/(rates['size']+DT_PRIOR_N)
    rates.assign(score=DT_SCORES[method]).to_csv(DT_OUT/f'fit_support_{method}.csv')
dt_rules=pd.DataFrame(rule_rows);dt_rules.to_csv(DT_OUT/'rules.csv',index=False)
display(dt_rules)
print('Số nhóm thực tế:',{m:dt_assign(dt_fit,m).nunique() for m in DT_METHODS})


In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
pd.set_option('display.max_seq_items', None)

In [ ]:
dt_metrics=[];dt_profiles=[];dt_coverage=[];dt_features=[];dt_transitions=[];dt_previous={}
for month in EVAL_MONTHS+[RECOMMEND_MONTH]:
    d,coverage=dt_common(month);dt_coverage.append(coverage)
    has_label=month in EVAL_MONTHS
    valid,y=target(d,month) if has_label else (np.zeros(len(d),bool),np.zeros(len(d),int))
    baseline=float(y[valid].mean()) if valid.any() else np.nan
    assignments=d[['customer_id']].copy();assignments['as_of_month']=month
    for method in DT_METHODS:
        keys=dt_assign(d,method);assignments[method+'_group']=keys
        fitted=keys.map(DT_SCORES[method]);score=fitted.fillna(DT_PRIORS[method]).to_numpy(float)
        if valid.any():dt_metrics.append(dict(month=month,method=method,fallback_share=float(fitted[valid].isna().mean()),**rfm_metrics(y[valid],score[valid])))
        for group,ix in keys.groupby(keys).groups.items():
            ix=np.asarray(ix);labelix=ix[valid[ix]];nn=len(labelix);positive=int(y[labelix].sum())
            lo,hi=wilson(np.array([positive]),np.array([nn]))
            rate=positive/nn if nn else np.nan
            dt_profiles.append(dict(month=month,method=method,group=group,n=len(ix),share=len(ix)/len(d),label_n=nn,positive_n=positive,
                rate=rate,ci_low=lo[0],ci_high=hi[0],baseline=baseline,lift=rate/baseline if baseline>0 else np.nan))
            for feature in ['F_event_months','M_group_points']:
                values=d.loc[ix,feature];values=values.loc[d.loc[ix,'F_event_months'].gt(0)] if feature=='R' else values
                dt_features.append(dict(month=month,method=method,group=group,feature=feature,n_valid=int(values.notna().sum()),
                    min=values.min(),p25=values.quantile(.25),median=values.median(),p75=values.quantile(.75),max=values.max()))
        curr=pd.DataFrame(dict(customer_id=d.customer_id,group=keys))
        if method in dt_previous:
            pm,prev=dt_previous[method];j=prev.merge(curr,on='customer_id',suffixes=('_from','_to'),validate='one_to_one')
            t=j.groupby(['group_from','group_to']).size().rename('n').reset_index()
            t['source_month']=pm;t['target_month']=month;t['method']=method;dt_transitions.append(t)
        dt_previous[method]=(month,curr)
    assignments.to_parquet(DT_OUT/f'fm_tree_assignments_{month}.parquet',index=False)
    # Retain excluded customers explicitly instead of silently assigning a score.
    original,_=load(month)
    status=original[['customer_id']].merge(assignments,on='customer_id',how='left',validate='one_to_one')
    status['as_of_month']=month;status['fm_tree_status']=np.where(status.FM_tree_group.notna(),'Assigned','ExcludedInvalidM')
    status.to_parquet(DT_OUT/f'fm_tree_coverage_{month}.parquet',index=False)
dt_metrics=pd.DataFrame(dt_metrics);dt_profiles=pd.DataFrame(dt_profiles);dt_features=pd.DataFrame(dt_features)
assert dt_metrics.groupby('month').n.nunique().eq(1).all()
dt_metrics.to_csv(DT_OUT/'metrics_monthly.csv',index=False);dt_profiles.to_csv(DT_OUT/'group_acquisition_monthly.csv',index=False)
dt_features.to_csv(DT_OUT/'feature_ranges_monthly.csv',index=False);pd.DataFrame(dt_coverage).to_csv(DT_OUT/'coverage_monthly.csv',index=False)
if dt_transitions:pd.concat(dt_transitions,ignore_index=True).to_csv(DT_OUT/'transitions.csv',index=False)
summary=dt_metrics.groupby('method').agg(AP_mean=('AP','mean'),Lift10_mean=('Lift10','mean'),Lift10_min=('Lift10','min'),Brier_mean=('Brier','mean'),fallback_max=('fallback_share','max'))
summary.to_csv(DT_OUT/'comparison_summary.csv');display(summary.round(5))
display(dt_metrics.pivot(index='method',columns='month',values='Lift10').round(3))
# Horizontal tables; '-' is a formatted range, not a subtraction.
labelled=dt_profiles.loc[dt_profiles.label_n.gt(0)].copy()
labelled['rate_CI_range']=labelled.apply(lambda r:f'{r.ci_low:.2%} - {r.ci_high:.2%}',axis=1)
display(labelled.pivot(index=['method','group'],columns='month',values='rate_CI_range'))
labelled.pivot(index=['method','group'],columns='month',values='rate_CI_range').to_csv(DT_OUT/'monthly_acquisition_CI_ranges.csv')
stability=labelled.groupby(['method','group']).agg(months=('month','nunique'),min_n=('label_n','min'),rate_min=('rate','min'),rate_max=('rate','max'),share_min=('share','min'),share_max=('share','max'))
stability['rate_range']=stability.apply(lambda r:f'{r.rate_min:.2%} - {r.rate_max:.2%}',axis=1)
stability['width_pp']=100*(stability.rate_max-stability.rate_min)
stability.to_csv(DT_OUT/'group_stability.csv');display(stability)
overlap=[]
for (method,month),g in labelled.groupby(['method','month']):
    from itertools import combinations
    for (_,a),(_,b) in combinations(g.iterrows(),2):
        overlap.append(dict(method=method,month=month,group_a=a['group'],group_b=b['group'],
            CI_overlap=max(a.ci_low,b.ci_low)<=min(a.ci_high,b.ci_high)))
pd.DataFrame(overlap).to_csv(DT_OUT/'monthly_CI_overlap.csv',index=False)
joblib.dump(dict(models=DT_MODELS,leaf_maps=DT_MAPS,scores=DT_SCORES,priors=DT_PRIORS,edges=edges),DT_OUT/'fm_tree_bundle.joblib')
(DT_OUT/'config.json').write_text(json.dumps(dict(fit_month=FIT_MONTH,fit_label_month=shift(FIT_MONTH,1),eval_months=EVAL_MONTHS,
    hybrid_method='FM_F4_M5; frozen F/M edges from '+RFM_METHOD,max_leaves_per_branch=DT_MAX_LEAVES_PER_BRANCH,min_leaf_share=DT_MIN_LEAF_SHARE,
    no_event_policy='separate branch; R undefined',M_validity='must verify upstream as-of',scope='development; no automatic replacement of P1-P6'),indent=2))
print('RFM tree outputs:',DT_OUT)


### Đọc và quyết định sau khi chạy
So sánh AP/Lift@10%/Brier cùng cohort, quy mô và acquisition range từng tháng. Range CI95% là khoảng bất định của rate nhóm từng tháng; min–max trong stability là độ dao động rate qua tháng, không phải range xác suất cá nhân. CI overlap không phải kiểm định hai nhóm giống nhau. Khoảng feature là phân bố quan sát, còn rules.csv mới là biên quyết định; khoảng biên của các chiều riêng lẻ có thể chồng lấp dù lá cây không chồng lấp trong không gian đa chiều.

Không bắt buộc cây phải chọn cả R,F,M: xem feature trong rules.csv. Không chọn lại số lá theo bảng các tháng này rồi gọi đó là test độc lập. Nếu muốn tối ưu số nhóm cần thêm validation theo thời gian. P1–P6 và output cũ được giữ để đối chiếu, phần mới chỉ là ứng viên thay thế. Khi chốt thay mới cập nhật downstream 11–13; không coi leaf ID là thứ hạng giá trị hay tên persona business.

## 9. Chọn số nhóm FM tree: 5–15
Thử mọi cách phân bổ ngân sách lá giữa NoEvent và HasEvent với tổng 5..15 (mỗi nhánh ít nhất một nhóm). Tái sử dụng cây nhánh theo ngân sách để giảm chi phí. Cây có thể dừng sớm: báo requested_groups và actual_groups, không ép tách nếu dữ liệu không hỗ trợ.

Chọn bằng AP tháng **12/2015→01/2016**: trong các cấu hình có AP cách tốt nhất không quá 0.001, ưu tiên ít nhóm thực tế hơn, rồi Brier thấp hơn. Đây là quy tắc đơn giản hóa đã khai báo trước, không phải kiểm định thống kê. Báo cả Lift10 và quy mô lá. Khóa lựa chọn trước khi đánh giá 01–04/2016. Đây vẫn là development đã được xem trong dự án. Phần này chỉ dùng FM, F-only/Hybrid ở phần 8 làm đối chiếu; giữ nguyên các ngưỡng và score đã fit November.

Chỉ dùng F/M; baseline FM_F4_M5 cũng bỏ R. Số nhóm được chọn theo AP, chưa tối ưu trực tiếp độ ổn định rate.

In [ ]:
DT_K_OPTIONS=list(range(5,16))
DT_AP_TOLERANCE=.001
branch_cache={};scan_specs=[]
for branch,mask in [('NoEvent',dt_fit.F_event_months.eq(0)),('HasEvent',dt_fit.F_event_months.gt(0))]:
    columns=['M_group_points'] if branch=='NoEvent' else ['F_event_months','M_group_points']
    train=mask.to_numpy()&dt_ok
    for budget in range(1,15):
        tree=None
        if budget>1:
            tree=DecisionTreeClassifier(max_leaf_nodes=budget,min_samples_leaf=max(100,int(np.ceil(train.sum()*DT_MIN_LEAF_SHARE))),random_state=42)
            tree.fit(dt_fit.loc[train,columns],dt_y[train])
        branch_cache[(branch,budget)]=dict(tree=tree,features=columns)
for k in DT_K_OPTIONS:
    for no_budget in range(1,k):
        has_budget=k-no_budget;method=f'FM_K{k:02d}_N{no_budget:02d}_H{has_budget:02d}'
        for branch,budget in [('NoEvent',no_budget),('HasEvent',has_budget)]:
            spec=branch_cache[(branch,budget)];DT_MODELS[(method,branch)]=spec
            tree=spec['tree']
            if tree is not None:
                leaves=np.flatnonzero(tree.tree_.children_left==-1)
                DT_MAPS[(method,branch)]={int(leaf):f'{method}_{branch}_G{i+1}' for i,leaf in enumerate(leaves)}
        keys=dt_assign(dt_fit,method)
        rates=pd.DataFrame(dict(group=keys,y=dt_y,valid=dt_ok)).query('valid').groupby('group').y.agg(['size','sum'])
        DT_PRIORS[method]=float(dt_y[dt_ok].mean())
        DT_SCORES[method]=(rates['sum']+DT_PRIOR_N*DT_PRIORS[method])/(rates['size']+DT_PRIOR_N)
        scan_specs.append(dict(method=method,requested_groups=k,no_event_budget=no_budget,has_event_budget=has_budget,
            actual_groups=keys.nunique(),fit_min_group_n=int(rates['size'].min())))
scan_specs=pd.DataFrame(scan_specs)
scan_val,scan_val_coverage=dt_common('2015-12');scan_valid,scan_y=target(scan_val,'2015-12')
validation_rows=[]
for spec in scan_specs.to_dict('records'):
    method=spec['method'];keys=dt_assign(scan_val,method);score=keys.map(DT_SCORES[method]).fillna(DT_PRIORS[method]).to_numpy(float)
    validation_rows.append(dict(**spec,**rfm_metrics(scan_y[scan_valid],score[scan_valid])))
scan_validation=pd.DataFrame(validation_rows)
scan_validation.to_csv(DT_OUT/'k_scan_validation_all_allocations.csv',index=False)
per_k=scan_validation.sort_values(['AP','Brier','method'],ascending=[False,True,True]).groupby('requested_groups',sort=True).head(1).sort_values('requested_groups')
per_k.to_csv(DT_OUT/'k_scan_validation_best_per_k.csv',index=False);display(per_k.round(5))
best_ap=scan_validation.AP.max()
shortlist=scan_validation.loc[scan_validation.AP.ge(best_ap-DT_AP_TOLERANCE)]
chosen=shortlist.sort_values(['actual_groups','Brier','requested_groups','method']).iloc[0]
SCAN_SELECTED=str(chosen['method'])
(DT_OUT/'k_selection.json').write_text(json.dumps(dict(selected=SCAN_SELECTED,requested_groups=int(chosen.requested_groups),
    actual_groups=int(chosen.actual_groups),validation_month='2015-12',AP_tolerance=DT_AP_TOLERANCE,
    criterion='within AP tolerance; fewer actual groups then lower Brier',fit_month=FIT_MONTH),indent=2))
print('Khóa trước evaluation:',SCAN_SELECTED,'; số nhóm thực tế:',int(chosen.actual_groups))


In [ ]:
# Compare one locked allocation per requested k; never reselect on these months.
scan_methods=list(dict.fromkeys(per_k.method.tolist()+[SCAN_SELECTED]));scan_rows=[];scan_profiles=[];scan_ranges=[];scan_moves=[];scan_prev={}
for month in ['2016-01','2016-02','2016-03','2016-04',RECOMMEND_MONTH]:
    d,cover=dt_common(month)
    valid,y=target(d,month) if month!=RECOMMEND_MONTH else (np.zeros(len(d),bool),np.zeros(len(d),int))
    assigned=d[['customer_id']].copy();assigned['as_of_month']=month
    for method in scan_methods+['FM_F4_M5']:
        keys=dt_assign(d,method);raw=keys.map(DT_SCORES[method]);score=raw.fillna(DT_PRIORS[method]).to_numpy(float)
        if valid.any():scan_rows.append(dict(month=month,method=method,selected=method==SCAN_SELECTED,fallback_share=float(raw[valid].isna().mean()),**rfm_metrics(y[valid],score[valid])))
        for group,ix in keys.groupby(keys).groups.items():
            ix=np.asarray(ix);jj=ix[valid[ix]];nn=len(jj);positive=int(y[jj].sum());lo,hi=wilson(np.array([positive]),np.array([nn]))
            scan_profiles.append(dict(month=month,method=method,group=group,n=len(ix),share=len(ix)/len(d),label_n=nn,positive_n=positive,
                rate=positive/nn if nn else np.nan,ci_low=lo[0],ci_high=hi[0]))
            if method==SCAN_SELECTED:
                for feature in ['F_event_months','M_group_points']:
                    v=d.loc[ix,feature]
                    if feature=='R':v=v.loc[d.loc[ix,'F_event_months'].gt(0)]
                    scan_ranges.append(dict(month=month,group=group,feature=feature,min=v.min(),p25=v.quantile(.25),median=v.median(),p75=v.quantile(.75),max=v.max()))
        curr=pd.DataFrame(dict(customer_id=d.customer_id,group=keys))
        if method in scan_prev:
            pm,prev=scan_prev[method];joined=prev.merge(curr,on='customer_id',suffixes=('_from','_to'),validate='one_to_one')
            scan_moves.append(dict(method=method,source_month=pm,target_month=month,common_n=len(joined),stay_share=float(joined.group_from.eq(joined.group_to).mean())))
        scan_prev[method]=(month,curr)
        if method==SCAN_SELECTED:assigned['fm_tree_group']=keys;assigned['fm_tree_score']=score
    assigned.to_parquet(DT_OUT/f'selected_fm_tree_{month}.parquet',index=False)
    print('K scan evaluation:',month)
scan_metrics=pd.DataFrame(scan_rows);scan_profiles=pd.DataFrame(scan_profiles)
assert scan_metrics.groupby('month').n.nunique().eq(1).all()
scan_metrics.to_csv(DT_OUT/'k_scan_evaluation_monthly.csv',index=False)
scan_profiles.to_csv(DT_OUT/'k_scan_group_rates_monthly.csv',index=False)
pd.DataFrame(scan_ranges).to_csv(DT_OUT/'selected_feature_ranges.csv',index=False)
pd.DataFrame(scan_moves).to_csv(DT_OUT/'k_scan_transition_stability.csv',index=False)
ss=scan_metrics.groupby('method').agg(AP_mean=('AP','mean'),Lift10_mean=('Lift10','mean'),Lift10_min=('Lift10','min'),Brier_mean=('Brier','mean'))
ss=ss.join(scan_specs.set_index('method')[['requested_groups','actual_groups']],how='left')
sp=scan_profiles.loc[scan_profiles.label_n.gt(0)].groupby(['method','group']).agg(rate_min=('rate','min'),rate_max=('rate','max'),min_n=('label_n','min'),min_share=('share','min'))
sp['width_pp']=100*(sp.rate_max-sp.rate_min);sp.to_csv(DT_OUT/'k_scan_group_stability.csv')
ss=ss.join(sp.groupby('method').agg(max_rate_width_pp=('width_pp','max'),smallest_group_n=('min_n','min'),smallest_share=('min_share','min')))
ss['selected_on_validation']=ss.index==SCAN_SELECTED
ss.to_csv(DT_OUT/'k_scan_summary.csv');display(ss.sort_values('requested_groups').round(5))
selected_profile=scan_profiles.loc[scan_profiles.method.eq(SCAN_SELECTED)&scan_profiles.label_n.gt(0)].copy()
selected_profile['range']=selected_profile.apply(lambda r:f'{r.ci_low:.2%} - {r.ci_high:.2%}',axis=1)
table=selected_profile.pivot(index='group',columns='month',values='range')
table.to_csv(DT_OUT/'selected_monthly_CI_ranges.csv');display(table)
# Export exact rules for the chosen configuration.
selected_rules=[]
for branch in ['NoEvent','HasEvent']:
    spec=DT_MODELS[(SCAN_SELECTED,branch)];tree=spec['tree'];columns=spec['features']
    base=['F_event_months = 0' if branch=='NoEvent' else 'F_event_months >= 1']
    if tree is None:selected_rules.append(dict(group=SCAN_SELECTED+'_'+branch+'_G1',rule=' AND '.join(base)))
    else:
        def visit(node,path):
            left,right=tree.tree_.children_left[node],tree.tree_.children_right[node]
            if left==right:selected_rules.append(dict(group=DT_MAPS[(SCAN_SELECTED,branch)][node],rule=' AND '.join(path)))
            else:
                f=columns[tree.tree_.feature[node]];t=tree.tree_.threshold[node]
                visit(left,path+[f'{f} <= {t:.9g}']);visit(right,path+[f'{f} > {t:.9g}'])
        visit(0,base)
selected_rules=pd.DataFrame(selected_rules);selected_rules.to_csv(DT_OUT/'selected_rules.csv',index=False);display(selected_rules)
joblib.dump(dict(selected=SCAN_SELECTED,models={b:DT_MODELS[(SCAN_SELECTED,b)] for b in ['NoEvent','HasEvent']},
    maps={b:DT_MAPS.get((SCAN_SELECTED,b),{}) for b in ['NoEvent','HasEvent']},scores=DT_SCORES[SCAN_SELECTED],prior=DT_PRIORS[SCAN_SELECTED]),DT_OUT/'selected_fm_tree_bundle.joblib')
print('Đề xuất theo validation:',SCAN_SELECTED,'; đọc độ ổn định evaluation trước khi quyết định triển khai.')
